# Mutual Fund Analytics - Day 4: Advanced Financial Analytics & Predictions

Welcome to the **Day 4 Notebook**! Today, we will explore advanced financial analytics and predictive forecasting. We will calculate rolling returns, Simple Moving Averages (SMAs), Bollinger Bands, historical drawdowns, and build an interactive compounding simulator to model SIP and lumpsum wealth trajectories. We will also apply polynomial regression to forecast NAV prices for the next 30 days.

### Step 1: Environment Setup
We add the project root to Python's system path to access modules and load packages like `pandas`, `sqlite3`, `matplotlib`, and `seaborn`.

In [ ]:
import sys
from pathlib import Path

# Find project root
notebook_dir = Path.cwd()
project_root = notebook_dir.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns

# Import advanced models from scripts.predictive_analysis
from scripts.predictive_analysis import (
    calculate_sip_growth,
    calculate_lumpsum_growth,
    calculate_bollinger_bands,
    calculate_drawdowns,
    calculate_rolling_returns,
    forecast_nav_trend
)

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.sans-serif'] = 'Inter'
plt.rcParams['font.family'] = 'sans-serif'
print("Advanced analytics modules loaded successfully!")

### Step 2: Establish Database Connection
We open a connection to the SQLite database `mutual_fund_analytics.db` containing our clean data tables.

In [ ]:
db_path = project_root / "data" / "db" / "mutual_fund_analytics.db"
conn = sqlite3.connect(str(db_path))
conn.execute("PRAGMA foreign_keys = ON;")
print(f"Connected to database: {db_path}")

### Step 3: Load NAV History for Analysis
We select SBI Bluechip Fund (AMFI Code: `119551`) as our sample fund and load its daily NAV history.

In [ ]:
query = "SELECT date, nav FROM fact_nav WHERE amfi_code = 119551 ORDER BY date;"
df_nav = pd.read_sql_query(query, conn)
df_nav['date'] = pd.to_datetime(df_nav['date'])
print(f"Loaded {len(df_nav)} NAV records for SBI Bluechip Fund.")
display(df_nav.head())

### Step 4: Volatility Bands (Bollinger Bands)
Bollinger Bands consist of a Simple Moving Average (typically 20 days) and standard deviation lines above and below the SMA. They are used to measure the volatility of a fund's price and identify potential overbought or oversold conditions.

In [ ]:
df_bb = calculate_bollinger_bands(df_nav)

# Plot last 100 days for clarity
df_bb_recent = df_bb.tail(100)

plt.figure(figsize=(12, 6))
plt.plot(df_bb_recent['date'], df_bb_recent['nav'], label='Daily NAV', color='#1a73e8', linewidth=2.0)
plt.plot(df_bb_recent['date'], df_bb_recent['SMA'], label='20-Day SMA', color='#e0a800', linestyle='--')
plt.fill_between(df_bb_recent['date'], df_bb_recent['Lower_Band'], df_bb_recent['Upper_Band'], color='rgba(26,115,232,0.1)', label='Volatility Band (Bollinger)')
plt.title("Bollinger Bands & Volatility Channel (SBI Bluechip)", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Date")
plt.ylabel("NAV (INR)")
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()

### Step 5: Historical Drawdown Analysis
Drawdown measures the peak-to-trough decline of a fund during a specific period. It is represented as a percentage drop from the highest NAV (peak) observed up to that point. This helps investors understand potential loss scenarios and recovery times.

In [ ]:
df_dd = calculate_drawdowns(df_nav)

plt.figure(figsize=(12, 5))
plt.fill_between(df_dd['date'], df_dd['Drawdown_Pct'], 0, color='#ea4335', alpha=0.4, label='Drawdown %')
plt.title("Historical Drawdown Timeline (SBI Bluechip)", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Date")
plt.ylabel("Drawdown (% Drop from Peak)")
plt.legend()
plt.tight_layout()
plt.show()

max_drawdown = df_dd['Drawdown_Pct'].min()
print(f"Maximum Historical Drawdown: {max_drawdown:.2f}%")

### Step 6: Rolling Returns (Annualized)
Rolling returns calculate the annualized return of a fund over a specific rolling window (e.g. 252 business days or 1 year) shifted day-by-day. Unlike point-to-point returns, rolling returns provide a realistic look at historical investor holding experiences.

In [ ]:
df_rr = calculate_rolling_returns(df_nav, window=252)

plt.figure(figsize=(12, 5))
plt.plot(df_rr['date'], df_rr['Rolling_Return_Ann'] * 100, color='#34a853', linewidth=2.0, label='1-Year Rolling Return (Ann.)')
plt.title("1-Year Rolling Annualized Returns (SBI Bluechip)", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Date")
plt.ylabel("Annualized Return (%)")
plt.legend(loc='lower left')
plt.tight_layout()
plt.show()

### Step 7: NAV Price Trend Forecasting
We apply polynomial regression (degree 2) on the historical NAV price days to predict the potential price trajectory for the next 30 days.

In [ ]:
df_hist, df_fc = forecast_nav_trend(df_nav, forecast_days=30)

# Filter history to last 150 days for visualization clarity
df_hist_recent = df_hist.tail(150)

plt.figure(figsize=(12, 6))
plt.plot(df_hist_recent['date'], df_hist_recent['nav'], label='Historical NAV', color='#1a73e8', linewidth=2.0)
plt.plot(df_fc['date'], df_fc['nav_forecast'], label='Forecasted NAV (30 Days)', color='#ea4335', linestyle=':', linewidth=2.5)
plt.title("Historical NAV & 30-Day Predictive Trend (SBI Bluechip)", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Date")
plt.ylabel("NAV (INR)")
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()

### Step 8: Interactive Compounding Investment Simulator
We simulate a SIP investment of ₹5,000 monthly vs a lumpsum investment of ₹50,000 at a 12% expected annual return over 10 years to showcase compound growth.

In [ ]:
monthly_amt = 5000
lumpsum_amt = 50000
yield_rate = 12.0
horizon = 10

df_sip_sim = calculate_sip_growth(monthly_amt, yield_rate, horizon)
df_lump_sim = calculate_lumpsum_growth(lumpsum_amt, yield_rate, horizon)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot SIP growth
axes[0].plot(df_sip_sim['Year'], df_sip_sim['Future_Value'], label='SIP Maturity Value', color='#1a73e8', linewidth=2.5)
axes[0].plot(df_sip_sim['Year'], df_sip_sim['Invested_Amount'], label='Amount Invested', color='#5f6368', linestyle='--')
axes[0].fill_between(df_sip_sim['Year'], df_sip_sim['Invested_Amount'], df_sip_sim['Future_Value'], color='rgba(26,115,232,0.1)', label='Wealth Gained')
axes[0].set_title(f"SIP Compound Growth (₹{monthly_amt}/month @ {yield_rate}%)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Years")
axes[0].set_ylabel("Value (INR)")
axes[0].legend(loc="upper left")

# Plot Lumpsum growth
axes[1].plot(df_lump_sim['Year'], df_lump_sim['Future_Value'], label='Lumpsum Maturity Value', color='#34a853', linewidth=2.5)
axes[1].plot(df_lump_sim['Year'], df_lump_sim['Invested_Amount'], label='Amount Invested', color='#5f6368', linestyle='--')
axes[1].fill_between(df_lump_sim['Year'], df_lump_sim['Invested_Amount'], df_lump_sim['Future_Value'], color='rgba(52,168,83,0.1)', label='Wealth Gained')
axes[1].set_title(f"Lumpsum Compound Growth (₹{lumpsum_amt} @ {yield_rate}%)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Years")
axes[1].set_ylabel("Value (INR)")
axes[1].legend(loc="upper left")

plt.suptitle("SIP vs Lumpsum Compounding Growth Simulations", fontsize=15, fontweight="bold", y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
conn.close()
print("Database connection closed. Day 4 advanced analysis completed successfully!")